In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import scanpy as sc
import episcanpy as epi
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

import glob
import os

from pathlib import Path
from scipy import io
from helps import *

# Raw CM

## Peak & GEX

In [ ]:
adata=sc.read_h5ad("../TopicModelling/Datasets/BMMCMultiOme/RawFiles/GSE194122_openproblems_neurips2021_multiome_BMMC_processed.h5ad")
adata

In [ ]:
adata = adata[adata.obs["Samplename"] == "site1_donor1_multiome"]
adata

In [ ]:
rna = adata[:, adata.var["feature_types"] == "GEX"]
atac = adata[:, adata.var["feature_types"] == "ATAC"]

rna.obs["CellType"] = rna.obs["cell_type"]
atac.obs["CellType"] = atac.obs["cell_type"]

rna.X=rna.layers["counts"]
rna.layers["raw"]=rna.layers["counts"]
del rna.layers["counts"]
atac.X=atac.layers["counts"]
atac.layers["raw"]=atac.layers["counts"]
del atac.layers["counts"]

In [ ]:
atac.var["chr"]=[ind.split("-")[0] for ind in atac.var.index]
atac.var["start"]=[ind.split("-")[1] for ind in atac.var.index]
atac.var["stop"]=[ind.split("-")[2] for ind in atac.var.index]

atac = atac[:,atac.var[atac.var['chr'].str.match('chr')].index]
atac = atac[:,atac.var[~atac.var['chr'].str.match('chrM')].index]
atac.var.index=["_".join([ind.split("-")[0],ind.split("-")[1],ind.split("-")[2]]) for ind in atac.var.index]
atac.var.head()

In [ ]:
sc.pp.scrublet(rna)

In [ ]:
Path("Datasets/BMMCMultiOme/CM/").mkdir(parents=True, exist_ok=True)
rna.write("Datasets/BMMCMultiOme/CM/BMMCMultiOme_GEX_Raw.h5ad", compression="gzip")
atac.write("Datasets/BMMCMultiOme/CM/BMMCMultiOme_Peak_Raw.h5ad", compression="gzip")

## mRNA

In [ ]:
adata=sc.read_h5ad("Datasets/BMMCMultiOme/CM/BMMCMultiOme_GEX_Raw.h5ad")
adata

In [ ]:
names = ["chr", "source", "type", "start", "stop", "score", "strand", "frame", "attribute"]
gtf_file="../AnnotRef/hs/T2T/GCF_009914755.1_T2T-CHM13v2.0_genomic.gtf.gz"
gtf=pd.read_csv(gtf_file, sep="\t", header=None, comment="#", names=names, dtype={"chr": str})
gtf=gtf[(gtf["type"]=="gene") & (gtf["source"]=="BestRefSeq")]

gene_names=[attr.replace("gene_id", "").strip().strip("\"") for feature_attr in gtf.attribute for attr in feature_attr.split(";") if attr.strip().startswith("gene_id")]
tmp = [attr.replace("gene_type", "").strip().strip("\"") for feature_attr in gtf.attribute for attr in feature_attr.split(";") if attr.strip().startswith("gene_type ")]
if not tmp:
    tmp = [attr.replace("gene_biotype", "").strip().strip("\"") for feature_attr in gtf.attribute for attr in feature_attr.split(";") if attr.strip().startswith("gene_biotype ")]

info=pd.DataFrame(data=[gene_names,tmp], index=["GeneName","GeneType"]).T
info.head()

mRNA=adata[:, intersection([adata.var.index, info[info["GeneType"]=="protein_coding"]["GeneName"]])]
mRNA.X=scipy.sparse.csr_matrix(mRNA.X.copy())
mRNA.layers["raw"]=scipy.sparse.csr_matrix(mRNA.X.copy())
mRNA

In [ ]:
mRNA.write("Datasets/BMMCMultiOme/CM/BMMCMultiOme_mRNA_Raw.h5ad", compression="gzip")

## lncRNA

In [ ]:
adata=sc.read_h5ad("Datasets/BMMCMultiOme/CM/BMMCMultiOme_GEX_Raw.h5ad")
adata

In [ ]:
names = ["chr", "source", "type", "start", "stop", "score", "strand", "frame", "attribute"]
gtf_file="../AnnotRef/hs/T2T/GCF_009914755.1_T2T-CHM13v2.0_genomic.gtf.gz"
gtf=pd.read_csv(gtf_file, sep="\t", header=None, comment="#", names=names, dtype={"chr": str})
gtf=gtf[(gtf["type"]=="gene") & (gtf["source"]=="BestRefSeq")]

gene_names=[attr.replace("gene_id", "").strip().strip("\"") for feature_attr in gtf.attribute for attr in feature_attr.split(";") if attr.strip().startswith("gene_id")]
tmp = [attr.replace("gene_type", "").strip().strip("\"") for feature_attr in gtf.attribute for attr in feature_attr.split(";") if attr.strip().startswith("gene_type ")]
if not tmp:
    tmp = [attr.replace("gene_biotype", "").strip().strip("\"") for feature_attr in gtf.attribute for attr in feature_attr.split(";") if attr.strip().startswith("gene_biotype ")]

info=pd.DataFrame(data=[gene_names,tmp], index=["GeneName","GeneType"]).T
info.head()

lncRNA=adata[:, intersection([adata.var.index, info[info["GeneType"]=="lncRNA"]["GeneName"]])]
lncRNA.X=scipy.sparse.csr_matrix(lncRNA.X.copy())
lncRNA.layers["raw"]=scipy.sparse.csr_matrix(lncRNA.X.copy())
lncRNA

In [ ]:
lncRNA.write("Datasets/BMMCMultiOme/CM/BMMCMultiOme_lncRNA_Raw.h5ad", compression="gzip")

# Processing

## Peak

In [ ]:
adata=sc.read_h5ad("Datasets/BMMCMultiOme/CM/BMMCMultiOme_Peak_Raw.h5ad")
adata.layers["raw"]=scipy.sparse.csr_matrix(adata.X.copy())
del adata.layers["counts"]
adata

In [ ]:
epi.pp.qc_stats(adata, verbose=True)

In [ ]:
min_features = 10**np.quantile(adata.obs["log_n_features"], 0.05)
max_features = 10**np.quantile(adata.obs["log_n_features"], 0.95)
min_cells = 10**np.quantile(adata.var["log_n_cells"], 0.05)
max_cells = 10**np.quantile(adata.var["log_n_cells"], 0.95)

In [ ]:
epi.pl.histogram(adata, "n_features", min_threshold=min_features, max_threshold=max_features, show_log=True)

In [ ]:
epi.pl.histogram(adata, "n_cells", min_threshold=min_cells, max_threshold=max_cells, show_log=True)

In [ ]:
epi.pp.set_filter(adata, "n_features", min_threshold=min_features, max_threshold=max_features, verbose=False)
epi.pp.set_filter(adata, "n_cells", min_threshold=min_cells, max_threshold=max_cells, verbose=False)
adata = epi.pp.apply_filters(adata, verbose=False)
adata

In [ ]:
epi.pp.normalize_total(adata)

In [ ]:
sc.pp.highly_variable_genes(adata, flavor='seurat_v3')

In [ ]:
min_var = np.quantile(adata.var.variances_norm, 0.7)

In [ ]:
adata=adata[:, adata.var.variances_norm>min_var]

In [ ]:
adata = adata[:, adata.X.max(axis=0)>0]
adata = adata[adata.X.max(axis=1)>0]

In [ ]:
adata

In [ ]:
adata.write("Datasets/BMMCMultiOme/CM/BMMCMultiOme_Peak_Def.h5ad", compression="gzip")

## GEX

In [ ]:
adata=sc.read_h5ad("Datasets/BMMCMultiOme/CM/BMMCMultiOme_GEX_Raw.h5ad")
adata.layers["raw"]=scipy.sparse.csr_matrix(adata.X.copy())
del adata.layers["counts"]
adata

In [ ]:
epi.pp.qc_stats(adata, verbose=False)
mt_gene_mask = np.flatnonzero([gene.startswith(("MT","mt")) for gene in adata.var.index])
adata.obs['mt_frac'] = np.sum(adata[:, mt_gene_mask].X, axis=1).A1/adata.obs['n_counts']

In [ ]:
min_features = 10**np.quantile(adata.obs["log_n_features"], 0.05)
max_features = 10**np.quantile(adata.obs["log_n_features"], 0.95)
min_cells = 10**np.quantile(adata.var["log_n_cells"], 0.05)
max_cells = 10**np.quantile(adata.var["log_n_cells"], 0.95)

In [ ]:
epi.pl.histogram(adata, "n_features", min_threshold=min_features, max_threshold=max_features, show_log=True)

In [ ]:
epi.pl.histogram(adata, "n_cells", min_threshold=min_cells, max_threshold=max_cells, show_log=True)

In [ ]:
epi.pl.histogram(adata, "mt_frac", max_threshold=0.05, show_log=False)

In [ ]:
epi.pp.set_filter(adata, "n_features", min_threshold=min_features, max_threshold=max_features, verbose=False)
epi.pp.set_filter(adata, "n_cells", min_threshold=min_cells, max_threshold=max_cells, verbose=False)
adata = epi.pp.apply_filters(adata, verbose=False)
adata = adata[adata.obs["mt_frac"]<=0.05]
adata=adata[adata.obs.predicted_doublet==False]
adata

In [ ]:
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)

In [ ]:
sc.pp.highly_variable_genes(adata, flavor='seurat_v3')
min_var = np.quantile(adata.var.variances_norm, 0.7)
adata=adata[:, adata.var.variances_norm>min_var]
adata = adata[:, adata.X.max(axis=0)>0]
adata = adata[adata.X.max(axis=1)>0]
adata

In [ ]:
adata.write("Datasets/BMMCMultiOme/CM/BMMCMultiOme_GEX_Def.h5ad", compression="gzip")

## mRNA

In [ ]:
adata=sc.read_h5ad("Datasets/BMMCMultiOme/CM/BMMCMultiOme_mRNA_Raw.h5ad")
adata.layers["raw"]=scipy.sparse.csr_matrix(adata.X.copy())
del adata.layers["counts"]
adata

In [ ]:
epi.pp.qc_stats(adata, verbose=False)
mt_gene_mask = np.flatnonzero([gene.startswith(("MT","mt")) for gene in adata.var.index])
adata.obs['mt_frac'] = np.sum(adata[:, mt_gene_mask].X, axis=1).A1/adata.obs['n_counts']

In [ ]:
min_features = 10**np.quantile(adata.obs["log_n_features"], 0.05)
max_features = 10**np.quantile(adata.obs["log_n_features"], 0.95)
min_cells = 10**np.quantile(adata.var["log_n_cells"], 0.05)
max_cells = 10**np.quantile(adata.var["log_n_cells"], 0.95)

In [ ]:
epi.pl.histogram(adata, "n_features", min_threshold=min_features, max_threshold=max_features, show_log=True)

In [ ]:
epi.pl.histogram(adata, "n_cells", min_threshold=min_cells, max_threshold=max_cells, show_log=True)

In [ ]:
epi.pl.histogram(adata, "mt_frac", max_threshold=0.05, show_log=False)

In [ ]:
epi.pp.set_filter(adata, "n_features", min_threshold=min_features, max_threshold=max_features, verbose=False)
epi.pp.set_filter(adata, "n_cells", min_threshold=min_cells, max_threshold=max_cells, verbose=False)
adata = epi.pp.apply_filters(adata, verbose=False)
adata = adata[adata.obs["mt_frac"]<=0.05]
adata=adata[adata.obs.predicted_doublet==False]
adata

In [ ]:
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)

In [ ]:
sc.pp.highly_variable_genes(adata, flavor='seurat_v3')
min_var = np.quantile(adata.var.variances_norm, 0.7)
adata=adata[:, adata.var.variances_norm>min_var]
adata = adata[:, adata.X.max(axis=0)>0]
adata = adata[adata.X.max(axis=1)>0]
adata

In [ ]:
adata.write("Datasets/BMMCMultiOme/CM/BMMCMultiOme_mRNA_Def.h5ad", compression="gzip")

## lncRNA

In [ ]:
adata=sc.read_h5ad("Datasets/BMMCMultiOme/CM/BMMCMultiOme_lncRNA_Raw.h5ad")
adata.layers["raw"]=scipy.sparse.csr_matrix(adata.X.copy())
del adata.layers["counts"]
adata

In [ ]:
epi.pp.qc_stats(adata, verbose=False)
mt_gene_mask = np.flatnonzero([gene.startswith(("MT","mt")) for gene in adata.var.index])
adata.obs['mt_frac'] = np.sum(adata[:, mt_gene_mask].X, axis=1).A1/adata.obs['n_counts']

In [ ]:
min_features = 10**np.quantile(adata.obs["log_n_features"], 0.05)
max_features = 10**np.quantile(adata.obs["log_n_features"], 0.95)
min_cells = 10**np.quantile(adata.var["log_n_cells"], 0.05)
max_cells = 10**np.quantile(adata.var["log_n_cells"], 0.95)

In [ ]:
epi.pl.histogram(adata, "n_features", min_threshold=min_features, max_threshold=max_features, show_log=True)

In [ ]:
epi.pl.histogram(adata, "n_cells", min_threshold=min_cells, max_threshold=max_cells, show_log=True)

In [ ]:
epi.pp.set_filter(adata, "n_features", min_threshold=min_features, max_threshold=max_features, verbose=False)
epi.pp.set_filter(adata, "n_cells", min_threshold=min_cells, max_threshold=max_cells, verbose=False)
adata = epi.pp.apply_filters(adata, verbose=False)
adata=adata[adata.obs.predicted_doublet==False]
adata

In [ ]:
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)

In [ ]:
sc.pp.highly_variable_genes(adata, flavor='seurat_v3')
min_var = np.quantile(adata.var.variances_norm, 0.1)
adata=adata[:, adata.var.variances_norm>min_var]
adata = adata[:, adata.X.max(axis=0)>0]
adata = adata[adata.X.max(axis=1)>0]
adata

In [ ]:
adata.write("Datasets/BMMCMultiOme/CM/BMMCMultiOme_lncRNA_Def.h5ad", compression="gzip")

## Muon objects

In [ ]:
import muon as mu
peak=sc.read_h5ad(f"Datasets/BMMCMultiOme/CM/BMMCMultiOme_Peak_Def.h5ad")
gex=sc.read_h5ad(f"Datasets/BMMCMultiOme/CM/BMMCMultiOme_GEX_Def.h5ad")
mrna=sc.read_h5ad(f"Datasets/BMMCMultiOme/CM/BMMCMultiOme_mRNA_Def.h5ad")
lncrna=sc.read_h5ad(f"Datasets/BMMCMultiOme/CM/BMMCMultiOme_lncRNA_Def.h5ad")

#Peak + GEX
print("Peak + GEX")
inter=intersection([peak.obs.index, gex.obs.index])
mudata=mu.MuData({"Peak" : peak[inter], "GEX" : gex[inter]})
mudata.write_h5mu(f"Datasets/BMMCMultiOme/CM/BMMCMultiOme_Peak_GEX_Def.h5mu", compression="gzip")

#Peak + mRNA
print("Peak + mRNA")
inter=intersection([peak.obs.index, mrna.obs.index])
mudata=mu.MuData({"Peak" : peak[inter], "mRNA" : mrna[inter]})
mudata.write_h5mu(f"Datasets/BMMCMultiOme/CM/BMMCMultiOme_Peak_mRNA_Def.h5mu", compression="gzip")

#Peak + lncRNA
print("Peak + lncRNA")
inter=intersection([peak.obs.index, lncrna.obs.index])
mudata=mu.MuData({"Peak" : peak[inter], "lncRNA" : lncrna[inter]})
mudata.write_h5mu(f"Datasets/BMMCMultiOme/CM/BMMCMultiOme_Peak_lncRNA_Def.h5mu", compression="gzip")

#mRNA + lncRNA
print("mRNA + lncRNA")
inter=intersection([mrna.obs.index, lncrna.obs.index])
mudata=mu.MuData({"mRNA" : mrna[inter], "lncRNA" : lncrna[inter]})
mudata.write_h5mu(f"Datasets/BMMCMultiOme/CM/BMMCMultiOme_mRNA_lncRNA_Def.h5mu", compression="gzip")

#Peak + mRNA + lncRNA
print("Peak + mRNA + lncRNA")
inter=intersection([peak.obs.index, mrna.obs.index, lncrna.obs.index])
mudata=mu.MuData({"Peak" : peak[inter], "mRNA" : mrna[inter], "lncRNA" : lncrna[inter]})
mudata.write_h5mu(f"Datasets/BMMCMultiOme/CM/BMMCMultiOme_Peak_mRNA_lncRNA_Def.h5mu", compression="gzip")

In [ ]:
mudata

# CTA

In [ ]:
df=sc.read_h5ad("Datasets/BMMCMultiOme/CM/BMMCMultiOme_GEX_Raw.h5ad").obs
df.head()

In [ ]:
df=pd.DataFrame(df["CellType"])
df.head()

In [ ]:
df["CellType"]=df["CellType"].str.replace("B1 B","B_B1")
df["CellType"]=df["CellType"].str.replace("Naive CD20+ B","B_CD20_naive")
df["CellType"]=df["CellType"].str.replace("Transitional B","B_Trans")

df["CellType"]=df["CellType"].str.replace("CD14+ Mono","Mono_CD14")
df["CellType"]=df["CellType"].str.replace("CD16+ Mono","Mono_CD16")

df["CellType"]=df["CellType"].str.replace("CD4+ T activated","T_CD4_acti")
df["CellType"]=df["CellType"].str.replace("CD4+ T naive","T_CD4_naive")
df["CellType"]=df["CellType"].str.replace("CD8+ T","T_CD8")

df["CellType"]=df["CellType"].str.replace("cDC2","DC_2c")
df["CellType"]=df["CellType"].str.replace("pDC","DC_p")


df["CellType1"]=df["CellType"]
df["CellType1"]=df["CellType1"].str.replace("Mono_CD14","Mono")
df["CellType1"]=df["CellType1"].str.replace("Mono_CD16","Mono")

df["CellType1"]=df["CellType1"].str.replace("T_CD4_acti","T_CD4")
df["CellType1"]=df["CellType1"].str.replace("T_CD4_naive","T_CD4")

df["CellType1"]=df["CellType1"].str.replace("B_B1","B")
df["CellType1"]=df["CellType1"].str.replace("B_CD20_naive","B")
df["CellType1"]=df["CellType1"].str.replace("B_Trans","B")

df["CellType1"]=df["CellType1"].str.replace("DC_2c","DC")
df["CellType1"]=df["CellType1"].str.replace("DC_p","DC")


df["CellType2"]=df["CellType1"]
df["CellType2"]=df["CellType2"].str.replace("T_CD4","T")
df["CellType2"]=df["CellType2"].str.replace("T_CD4","T")
df["CellType2"]=df["CellType2"].str.replace("T_CD8","T")

In [ ]:
set(df["CellType2"])

In [ ]:
df[["CellType","CellType1","CellType2"]].to_csv("Datasets/BMMCMultiOme/BMMCMultiOme_Metadata.tsv.gz", sep="\t", compression="gzip")